# Working with Reward Pools and Asset Pools

This notebook demonstrates how to work with **RewardPool** and **AssetPool** assets in Anno 117.

## What are Pools?

Pools are containers that group items with associated probabilities:
- **RewardPool**: Contains items with weights for random selection (used by traders, quests, expeditions)
- **AssetPool**: Generic container for grouping assets (often equal probabilities)

Pools can contain other pools, creating hierarchies:
```
Festival Happiness Pool (root)
├─ Culture Pool (70% weight)
│  ├─ Item A (10% of Culture)
│  └─ Item B (5% of Culture)
├─ Diplomacy Pool (20% weight)
└─ Finance Pool (10% weight)
```

## Key Features

1. **`pool.pool_assets()`**: Get all items from a pool with their probabilities
2. **`item.in_reward_pool`**: Find which pools contain an item (reverse lookup)
3. **Automatic probability calculation**: Weights are normalized and multiplied through the hierarchy

## Setup

In [2]:
from assetextractor.extraction.utils import Config
from assetextractor.parsing.core.assets import AssetCache

# Load assets
print("Loading assets...")
config = Config.from_json("config.json")
assets = AssetCache.load(config)

print(f"Loaded {len(assets.elements)} assets")

Loading assets...
Loaded 28559 assets


## Part 1: Understanding Pool Structures

Let's explore a specific pool to understand its structure.

In [3]:
# Get a specific reward pool (Festival Happiness)
pool = assets[122533]

print(f"Pool: {pool.name}")
print(f"GUID: {pool.guid}")
print(f"Template: {pool.template.name}")
print()

# Check if it's referenced by non-pool assets (i.e., a "root pool")
is_root_pool = False
referencing_assets = []
for ref in pool.referenced_by.values():
    if "Pool" not in ref.source.template.name:
        is_root_pool = True
        referencing_assets.append(ref.source)

if is_root_pool:
    print("This is a ROOT POOL (referenced by non-pool assets):")
    for asset in referencing_assets[:5]:  # Show first 5
        print(f"  - {asset.template.name}: {asset.name}")
else:
    print("This is a SUBPOOL (only referenced by other pools)")

Pool: RewardPool Festival Happiness
GUID: 122533
Template: RewardPool

This is a ROOT POOL (referenced by non-pool assets):
  - Festival: Festival Happiness


## Part 2: Using `pool_assets()` to Get Items with Probabilities

The `pool_assets()` method recursively processes the pool tree and returns all leaf items with their cumulative probabilities.

In [4]:
# Get all items from the pool with their probabilities
pool_results = pool.pool_assets()

print(f"Found {len(pool_results)} items in pool '{pool.name}':\n")

# Sort by probability (descending)
sorted_items = sorted(pool_results.items(), key=lambda x: x[1], reverse=True)

# Show top 10 items
print("Top 10 most likely items:")
for asset, probability in sorted_items[:10]:
    print(f"  {probability * 100:5.2f}% - {asset.name} ({asset.guid})")

print(f"\nTotal probability: {sum(pool_results.values()):.6f} (should be ~1.0)")

Found 47 items in pool 'RewardPool Festival Happiness':

Top 10 most likely items:
  12.58% - Specialist Area R InstitutionRiot  (51442)
  12.58% - Specialist Area R Weapons  (51450)
  12.58% - Specialist Balance R Happiness (140407)
   8.01% - Specialist Area C Tunics  (51446)
   8.01% - Specialist Incident C Riot (54021)
   8.01% - Specialist Balance C Happiness (64954)
   6.86% - Specialist Balance C Prestige (64951)
   5.39% - Specialist Area R Glass  (51599)
   5.39% - Specialist Balance R Prestige (140388)
   3.43% - Specialist Maintenance E Artisans (51342)

Total probability: 1.000000 (should be ~1.0)


## Part 3: Finding Items by Their Reward Pools

Use `in_reward_pool` to find which pools contain a specific item (reverse lookup).

In [5]:
# Get a specific item
item = assets[51446]  # Specialist Area C Tunics

print(f"Item: {item.name} ({item.guid})")
print(f"Template: {item.template.name}")
print()

# Check which reward pools contain this item
if item.in_reward_pool:
    print(f"This item appears in {len(item.in_reward_pool)} reward pools:\n")
    
    # Sort by probability (descending)
    sorted_pools = sorted(item.in_reward_pool.items(), key=lambda x: x[1].weight, reverse=True)
    
    for pool_guid, ref in sorted_pools[:10]:  # Show top 10
        pool = ref.source
        probability = ref.weight
        print(f"  {probability * 100:5.2f}% - {pool.name} ({pool.guid})")
else:
    print("This item is not in any reward pools.")

Item: Specialist Area C Tunics  (51446)
Template: Item

This item appears in 67 reward pools:

   8.01% - RewardPool Festival Happiness (122533)
   8.01% - RewardPool Festival Cernunnos (122546)
   1.95% - RewardPool Tarragon Contracts Base (64769)
   1.95% - RewardPool Licia Contracts Base (64772)
   1.95% - RewardPool Zarai Contracts Base (64778)
   1.95% - RewardPool Concordia Contracts Base (64781)
   1.95% - RewardPool Nefeneru Contracts Base (64784)
   1.48% - RewardPool Athr Contracts Base (64775)
   1.48% - RewardPool Procurator Contracts Base (64796)
   1.48% - RewardPool Manx Contracts Base (64799)


## Part 4: Identifying Item Sources (Traders, Quests, Expeditions)

Now we can trace back from pools to their sources to find out where items can be obtained.

In [6]:
def find_item_sources(item, assets):
    """Find all sources (traders, quests, etc.) that offer this item."""
    sources = []
    
    # Go through all pools containing this item
    if item.in_reward_pool:
        for pool_guid, ref in item.in_reward_pool.items():
            pool = ref.source
            probability = ref.weight
            
            # Check what references this pool (should be non-pool assets since it's a root pool)
            for pool_ref in pool.referenced_by.values():
                source = pool_ref.source
                source_type = source.template.name
                
                sources.append({
                    'type': source_type,
                    'name': source.name if source.name else str(source.guid),
                    'guid': source.guid,
                    'pool': pool.name,
                    'probability': probability
                })
    
    return sources

# Find sources for our test item
sources = find_item_sources(item, assets)

if sources:
    print(f"Item '{item.name}' can be obtained from:\n")
    
    # Group by source type
    from collections import defaultdict
    by_type = defaultdict(list)
    for source in sources:
        by_type[source['type']].append(source)
    
    for source_type, source_list in sorted(by_type.items()):
        print(f"\n{source_type} ({len(source_list)}):")
        for source in source_list[:5]:  # Show first 5 per type
            print(f"  {source['probability'] * 100:5.2f}% - {source['name']} (via {source['pool']})")
else:
    print(f"No sources found for '{item.name}'")

Item 'Specialist Area C Tunics ' can be obtained from:


Achievement (2):
   0.33% - Achievement_Set2_08_House_9_Specialists (via RewardList Specialist Items)
   0.33% - Achievement_Set10_01_Specialist_In_Villa (via RewardList Specialist Items)

Festival (5):
   8.01% - Festival Happiness (via RewardPool Festival Happiness)
   1.14% - Festival Health (via RewardPool Festival Health)
   1.14% - Festival Knowledge (via RewardPool Festival Knowledge)
   1.14% - Festival Mercury Lugus (via RewardPool Festival Mercury Lugus)
   8.01% - Festival Cernunnos (via RewardPool Festival Cernunnos)

Function (2):
   0.33% - Find Island with ItemsInStock (via RewardList Specialist Items)
   0.33% - Find Island with ItemsInStock (via RewardList Specialist Items)

Participant 2ndParty (Rival) (28):
   1.45% - Rival_01 (Dorian) (via RewardPool Dorian Contracts Base)
   1.09% - Rival_01 (Dorian) (via RewardPool Dorian Contracts Good)
   0.00% - Rival_01 (Dorian) (via RewardPool Dorian Contracts Best)
   

## Part 5: Analyzing Pool Distribution and Rarity

Let's analyze the distribution of items across pools to understand rarity.

In [7]:
# Collect statistics about pool memberships
pool_counts = {}
max_probability = {}
total_probability = {}

for asset in assets.elements.values():
    if len(asset.in_reward_pool) > 0:
        pool_count = len(asset.in_reward_pool)
        pool_counts[asset.guid] = pool_count
        
        # Find max probability across all pools
        max_prob = max(ref.weight for ref in asset.in_reward_pool.values())
        max_probability[asset.guid] = max_prob
        
        # Sum probabilities across all pools
        total_prob = sum(ref.weight for ref in asset.in_reward_pool.values())
        total_probability[asset.guid] = total_prob

print(f"Items in reward pools: {len(pool_counts)}")
print()

# Items in most pools
print("Items appearing in most pools:")
top_by_count = sorted(pool_counts.items(), key=lambda x: x[1], reverse=True)[:10]
for guid, count in top_by_count:
    asset = assets[guid]
    print(f"  {count:3d} pools - {asset.name} ({guid})")

print()

# Items with highest single-pool probability
print("Items with highest probability in a single pool:")
top_by_prob = sorted(max_probability.items(), key=lambda x: x[1], reverse=True)[:10]
for guid, prob in top_by_prob:
    asset = assets[guid]
    print(f"  {prob * 100:5.2f}% - {asset.name} ({guid})")

Items in reward pools: 409

Items appearing in most pools:
   70 pools - Specialist Balance C Belief (64949)
   70 pools - Specialist Balance R Belief (140344)
   70 pools - Specialist Balance E Belief (64961)
   70 pools - Specialist Balance L Belief (80677)
   70 pools - Specialist Maintenance C Extractors (51330)
   70 pools - Specialist Maintenance R Hunters & Fisheries (51292)
   70 pools - Specialist Area E Togas (51390)
   70 pools - Specialist NeedReward E Oysters (51849)
   70 pools - Specialist NeedReward E Togas (51867)
   70 pools - Specialist PublicReward E SportingGrounds (42044)

Items with highest probability in a single pool:
  50.00% - Good Iron (2115)
  50.00% - Good Weapons (2173)
  33.33% - Good Porridge (2136)
  33.33% - Good Bread (2137)
  33.33% - Good Wine (2138)
  25.00% - Good Leather (2110)
  25.00% - Good Cloth (2121)
  25.00% - Good Cushions (8562)
  25.00% - Good Oysters with Caviar (2140)
  25.00% - Good Fine Glass (2151)


## Part 6: Exploring Pool Hierarchies

Let's visualize a pool hierarchy to understand how nested pools work.

In [8]:
def show_pool_structure(pool, max_depth=2, current_depth=0, cumulative_prob=1.0):
    """Recursively display pool structure with probabilities."""
    indent = "  " * current_depth
    
    if current_depth == 0:
        print(f"{pool.name} ({pool.guid})")
    
    if current_depth >= max_depth:
        return
    
    # Get entries from the pool
    entries = None
    is_reward_pool = False
    
    if "RewardPool" in pool.template.name:
        is_reward_pool = True
        try:
            entries = pool.RewardPool.ItemsPool
        except:
            pass
    elif "AssetPool" in pool.template.name:
        try:
            entries = pool.AssetPool.AssetList
        except:
            pass
    
    if not entries or len(entries._value_list) == 0:
        return
    
    # Calculate total weight
    total_weight = 0.0
    valid_entries = []
    
    for entry in entries:
        try:
            if is_reward_pool:
                ref = entry.ItemLink()
                weight = entry.Weight() if hasattr(entry, 'Weight') and entry.Weight() else 1.0
            else:
                ref = entry.Asset()
                weight = 1.0
            
            if ref:
                valid_entries.append((ref, weight))
                total_weight += weight
        except:
            pass
    
    # Show entries
    for ref, weight in sorted(valid_entries, key=lambda x: x[1], reverse=True)[:5]:  # Top 5
        local_prob = weight / total_weight if total_weight > 0 else 0
        new_cumulative = cumulative_prob * local_prob
        
        if "Pool" in ref.template.name:
            print(f"{indent}├─ [{local_prob * 100:5.2f}%] {ref.name} (Pool)")
            show_pool_structure(ref, max_depth, current_depth + 1, new_cumulative)
        else:
            print(f"{indent}├─ [{local_prob * 100:5.2f}%] [{new_cumulative * 100:5.2f}% total] {ref.name}")

# Show structure of our test pool
print("Pool hierarchy (max depth 2):\n")
show_pool_structure(pool, max_depth=2)

Pool hierarchy (max depth 2):

RewardPool Manx Contracts Base (64799)
├─ [78.43%] RewardList Manx Common (Pool)
  ├─ [ 1.85%] [ 1.45% total] Captain Durability C Speed
  ├─ [ 1.85%] [ 1.45% total] Specialist Productivity C EelChain
  ├─ [ 1.85%] [ 1.45% total] Specialist Productivity C SailsChain
  ├─ [ 1.85%] [ 1.45% total] Specialist Productivity C RopesChain
  ├─ [ 1.85%] [ 1.45% total] Specialist Productivity C BreadChain
├─ [19.61%] RewardList Manx Rare (Pool)
  ├─ [ 1.27%] [ 0.25% total] Captain Durability R Cargo
  ├─ [ 1.27%] [ 0.25% total] Specialist Productivity R CheeseChain
  ├─ [ 1.27%] [ 0.25% total] Specialist Productivity R ChariotsChain
  ├─ [ 1.27%] [ 0.25% total] Specialist Productivity R BroochesChain
  ├─ [ 1.27%] [ 0.25% total] Specialist Productivity R PeltHatsChain
├─ [ 0.98%] RewardList Manx Epic (Pool)
  ├─ [ 1.59%] [ 0.02% total] Specialist Military E Gates
  ├─ [ 1.59%] [ 0.02% total] Specialist Incident E Illness
  ├─ [ 1.59%] [ 0.02% total] Specialist Area

## Part 7: Finding All Traders Offering a Specific Item

Let's find all traders that might offer a specific item.

In [9]:
def get_english_name(asset):
    """Get English name from asset."""
    if asset.text and "english" in asset.text.values:
        return asset.text.values["english"]
    return asset.name if asset.name else f"Asset {asset.guid}"

# Find all traders
traders = []
if "Participant 3rdParty" in assets.templates:
    traders = list(assets.templates["Participant 3rdParty"].assets)

print(f"Found {len(traders)} traders\n")

# Check which traders offer our test item
traders_offering_item = []

for trader in traders:
    try:
        offered_items = trader.find("Trader.OfferedItems")
        if offered_items and offered_items():
            reward_pool = offered_items()
            
            # Check if our item is in this pool
            if reward_pool.guid in item.in_reward_pool:
                ref = item.in_reward_pool[reward_pool.guid]
                probability = ref.weight
                
                traders_offering_item.append({
                    'trader': trader,
                    'pool': reward_pool,
                    'probability': probability
                })
    except:
        pass

if traders_offering_item:
    print(f"Traders offering '{item.name}':\n")
    for data in sorted(traders_offering_item, key=lambda x: x['probability'], reverse=True):
        trader_name = get_english_name(data['trader'])
        probability = data['probability']
        print(f"  {probability * 100:5.2f}% - {trader_name}")
else:
    print(f"No traders found offering '{item.name}'")

Found 4 traders

Traders offering 'Specialist Area C Tunics ':

   1.07% - Procurator Corvinus
   1.07% - Manx
   1.05% - Diana
   1.05% - Valeria


## Part 8: Export Pool Data to CSV

Let's export pool probability data for further analysis.

In [10]:
import csv
from pathlib import Path

# Create output directory
output_dir = Path("../../../results/example")
output_dir.mkdir(parents=True, exist_ok=True)

# Export pool contents to CSV
output_file = output_dir / "pool_contents.csv"

with open(output_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Pool GUID', 'Pool Name', 'Item GUID', 'Item Name', 'Template', 'Probability %'])
    
    # Get all root reward pools
    root_pools = []
    for asset in assets.elements.values():
        if "RewardPool" in asset.template.name:
            # Check if it's a root pool
            is_root = False
            for ref in asset.referenced_by.values():
                if "Pool" not in ref.source.template.name:
                    is_root = True
                    break
            
            if is_root:
                root_pools.append(asset)
    
    # Export first 10 pools
    for pool in root_pools[:10]:
        pool_results = pool.pool_assets()
        
        for asset, probability in sorted(pool_results.items(), key=lambda x: x[1], reverse=True):
            writer.writerow([
                pool.guid,
                pool.name,
                asset.guid,
                asset.name if asset.name else f"Asset {asset.guid}",
                asset.template.name,
                f"{probability * 100:.4f}"
            ])

print(f"Exported pool contents to {output_file}")
print(f"Processed {len(root_pools[:10])} pools")

Exported pool contents to ..\..\..\results\example\pool_contents.csv
Processed 10 pools


## Summary

In this notebook, we learned:

1. **Pool structures**: RewardPool vs AssetPool, root pools vs subpools
2. **`pool_assets()`**: Get all items from a pool with probabilities
3. **`in_reward_pool`**: Reverse lookup to find which pools contain an item
4. **Item sources**: Trace pools back to traders, quests, and expeditions
5. **Pool analysis**: Understand distribution and rarity
6. **Pool hierarchies**: Visualize nested pool structures
7. **Practical queries**: Find traders offering specific items
8. **Data export**: Export pool data for external analysis

## Key Takeaways

- Probabilities are automatically calculated and accumulated through pool hierarchies
- Only "root pools" (referenced by non-pool assets) are tracked in `in_reward_pool`
- The same item can appear in multiple pools with different probabilities
- Use `pool_assets()` to get all items, `in_reward_pool` to find pools for an item